# Amparo -- M3, paso 7: corrida final (busqueda nueva y prompt de servicio)

Genera las respuestas que deciden si M3 llega a su meta
(`docs/m3_protocolo_aceptacion.md`): **articulo correcto en el top-5 en >= 28 de
45** y **>= 18 de 45 gold bien respondidas** con el juez del RAG. Solo genera; el
juez y RAGAS se corren despues en un entorno sin GPU (la key de Groq no hace
falta aqui).

**Cuatro sistemas, para que cada mejora se pueda atribuir a su causa** (una sola
variable entre vecinos):

| sistema | busqueda | prompt | que se genera aqui |
|---|---|---|---|
| **A** | produccion | el de entrenamiento | solo las 26 preguntas nuevas (las 75 ya estan: `results/m3_s10_2026-10-14/`, deterministas) |
| **A_servicio** | produccion | + bloque de servicio | 75 + 26 |
| **R** | la elegida en CPU (`results/m3_paso7_2026-10-14/busqueda_elegida.json`) | el de entrenamiento | 75 + 26 |
| **R_servicio** | la elegida en CPU | + bloque de servicio v1 | 75 + 26 |
| **R_servicio2** | la elegida en CPU | + bloque de servicio **v2** | 75 + 26 (segunda corrida) |

**Segunda corrida (2026-10-14).** La primera genero A, A_servicio, R y R_servicio;
R quedo adoptado en produccion y el bloque v1 no (casi duplico las abstenciones).
Al volver a correr, los cuatro primeros ya estan en Drive y se saltan: solo se
genera **R_servicio2** (unas 100 consultas, ~15 minutos). A y A_servicio piden
`use_rerank=False` explicito: desde que el reranker es produccion, A sin eso ya no
seria el control.

El bloque de servicio (`prompt_template.INSTRUCCIONES_DE_SERVICIO`) va **al
final** del system prompt con que se entreno, que queda intacto delante.

**Costo:** unos 330 consultas a ~8 s en A100: ~45 minutos. Cada respuesta se
guarda en Drive al generarse: si Colab se desconecta, se vuelve a correr y sigue
donde iba.

**Secretos:** ninguno. Necesita Drive (indice y adaptador v1).

In [ ]:
# Rama del repo a clonar.
RAMA = "m3.5"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/pipeline.py",
    "tools/rag/prompt_template.py",
    "tools/evaluation/preguntas_confirmacion.py",
    "data/eval_set.json",
    "data/preguntas_confirmacion_m3.json",
    "results/m3_paso7_2026-10-14/busqueda_elegida.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(f"Falta en la rama '{RAMA}': {faltan}. Hay que subir el repo antes de correr.")
print("Repo OK.")

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
# Versiones FIJADAS (H9), registradas por las corridas del 2026-10-09: el
# archivo dice de donde sale cada una. Antes era `pip install -U`.
!pip install -q -r colab/requirements-m3.txt

## Fase 1 -- Indice (verificado por huella) y generador (v1, verificado por huella)

In [ ]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

# Se verifica ANTES de medir, no solo se registra despues en el manifiesto: con
# otro indice las cifras no se comparan con las anteriores y nada falla a la
# vista. Las huellas son las del indice evaluado en S08, S10 y busqueda_v2.
from tools.auditoria_61 import verificar as _verificar_indice

_problemas = _verificar_indice(config.FAISS_INDEX_PATH, config.FAISS_METADATA_PATH)
if _problemas:
    raise SystemExit("El indice no es el evaluado; no se mide:\n  " + "\n  ".join(_problemas))
print("Indice verificado por huella.")

In [ ]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)
print(f"Indice denso: {len(store)} chunks")

USE_LORA = True
print("Adaptador verificado:", pipeline.verificar_adaptador())
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

## Fase 2 -- Los sistemas y las preguntas

In [ ]:
import json, pathlib
from tools.evaluation import eval_set as eval_set_mod
from tools.evaluation import preguntas_confirmacion

ELEGIDA = json.loads(pathlib.Path("results/m3_paso7_2026-10-14/busqueda_elegida.json").read_text(encoding="utf-8"))
print("Busqueda elegida en CPU:", ELEGIDA["nombre"], "--", ELEGIDA["motivo"])
BUSQUEDA_R = ELEGIDA["opciones"]          # argumentos de pipeline.answer_query

SISTEMAS = {
    "A": dict(use_rerank=False),                       # la produccion anterior, el control
    "A_servicio": dict(use_rerank=False, prompt_servicio=True),
    "R": dict(BUSQUEDA_R),
    "R_servicio": dict(BUSQUEDA_R, prompt_servicio=True),
    "R_servicio2": dict(BUSQUEDA_R, prompt_servicio="v2"),
}
CONJUNTOS = {
    "eval_set": eval_set_mod.load_eval_set(),
    "confirmacion": preguntas_confirmacion.cargar(),
}
# A sobre el eval set ya esta generado (S10 del 2026-10-14, determinista).
PLAN = [("A", "confirmacion")] + [(s, c) for s in ("A_servicio", "R", "R_servicio", "R_servicio2")
                                  for c in ("eval_set", "confirmacion")]
for s, c in PLAN:
    print(f"  {s:12} {c:13} {len(CONJUNTOS[c])} preguntas  {SISTEMAS[s]}")
print("Total:", sum(len(CONJUNTOS[c]) for _, c in PLAN), "consultas")

## Fase 3 -- Generar (retoma si se corta)

Cada respuesta se agrega a `rag/corridas_paso7/eval_records_<sistema>_<conjunto>.jsonl`
en Drive apenas se genera. Al volver a correr la celda, las ya hechas se saltan.

In [ ]:
import time
from tools.rag import config

destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_paso7"
destino.mkdir(parents=True, exist_ok=True)

for sistema, conjunto in PLAN:
    ruta = destino / f"eval_records_{sistema}_{conjunto}.jsonl"
    hechos = set()
    if ruta.exists():
        hechos = {json.loads(l)["id"] for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()}
    pendientes = [r for r in CONJUNTOS[conjunto] if r["id"] not in hechos]
    print(f"{sistema} / {conjunto}: {len(hechos)} hechas, {len(pendientes)} por generar")
    t_inicio = time.perf_counter()
    for i, reg in enumerate(pendientes, start=1):
        t0 = time.perf_counter()
        res = pipeline.answer_query(reg["messages"][1]["content"], store, use_lora=USE_LORA,
                                    bm25=bm25, model_bundle=model_bundle, **SISTEMAS[sistema])
        rec = pipeline.to_eval_record(res, reg)
        rec["sistema_paso7"] = sistema
        rec["segundos"] = round(time.perf_counter() - t0, 2)
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        if i % 10 == 0 or i == len(pendientes):
            prom = (time.perf_counter() - t_inicio) / i
            print(f"   {i}/{len(pendientes)} -- {prom:.1f} s/consulta, ETA ~{prom * (len(pendientes) - i) / 60:.0f} min")
print("Listo. Archivos en", destino)

In [ ]:
# Manifiesto de la corrida: commit, indice, adaptador, eval set, versiones, GPU.
import subprocess
from tools.rag import manifiesto

_hardware = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True).stdout.strip()
_m = manifiesto.construir("m3_paso7", use_lora=USE_LORA, n_chunks=len(store.metadata),
                          n_eval_set=len(CONJUNTOS["eval_set"]), hardware=_hardware)
print("Manifiesto:", _m.guardar(destino))
print(f"  commit {_m.git_commit[:8]} | indice {_m.hash_indice} | adaptador {_m.hash_adaptador} "
      f"| eval set {_m.hash_eval_set}")
with open(destino / "busqueda_elegida.json", "w", encoding="utf-8") as f:
    json.dump(ELEGIDA, f, ensure_ascii=False, indent=2)

---
Despues: desconectar la GPU. El juez del RAG, el acierto@5 sobre lo que de verdad
se recupero y RAGAS se corren en CPU con `tools/evaluation/juez_rag.py` y las
herramientas del paso 7 (`docs/m3_paso7_busqueda_2026-10-14.md`).

*Amparo -- M3 -- paso 7.*